In [1]:
import numpy as np
import utils
import torch
from torch import Tensor
from collections import Counter
import torch.nn.functional as F

upload_ids = ["001","002","003"]
upload_dir = "/mnt/g/uploads/"
models_dir = "/mnt/g/models/"
artifacts_dir = "/mnt/g/song_artifacts/"
model_filename = "0.pth"


from utils.model import AudioCNN  # Make sure the class definition is present

model = AudioCNN()
model.load_state_dict(torch.load("/mnt/g/models/audio_cnn_model.pt", map_location='cpu'))
model.eval()


for upload_id in upload_ids:
    
    filename = utils.find_file_by_upload_id(upload_id, upload_dir)
    signal, sr = utils.extract_signal_for_inference(filename)
    
    X = []
    preds = {}
    class_guesses = []
    
    for i in range(1,14,1):
        X.append(signal[i*22050:(i+1)*22050])
    
    for index, frame in enumerate(X):
        waveform = torch.from_numpy(frame).float().unsqueeze(0).unsqueeze(0)  # shape (1, 1, 22050)
        with torch.no_grad():
            output = model(waveform)
            prediction = F.softmax(output, dim=1) * 100
            
            preds[index] = (prediction.cpu().numpy())
            class_id = torch.argmax(output)
            class_guesses.append(utils.reverse_one_hot(class_id.item())) 

    mean_pred = np.zeros_like(next(iter(preds.values())))
    for pred in preds.values():
        mean_pred += pred
    mean_pred /= len(preds)

    all_preds = np.stack(list(preds.values()), axis=0).squeeze(1)
    median_pred = np.median(all_preds, axis=0) 
    
    counter = Counter(class_guesses)
    print(f"\n\nmean_pred: {mean_pred}\n\nprediction_medians: {median_pred}\n\nclass guesses: {counter}\n\n------")





mean_pred: [[26.59137   23.67995   20.244934  27.227161   2.2565868]]

prediction_medians: [30.20451   20.902737  19.835346  27.577084   1.7918153]

class guesses: Counter({'Rock': 8, 'Pop': 3, 'Hip-Hop': 2})

------


mean_pred: [[13.947295   4.713779   7.4309254 11.384822  62.523174 ]]

prediction_medians: [14.180109   4.60884    6.6274734 10.89788   63.892597 ]

class guesses: Counter({'Classical': 13})

------


mean_pred: [[27.746422   23.495699   21.448355   27.209543    0.09997547]]

prediction_medians: [29.03012   22.697731  20.755222  27.178596   0.0652231]

class guesses: Counter({'Rock': 8, 'Pop': 3, 'Hip-Hop': 2})

------
